# Section 1 : Environment Setup & Imports
Installing the required dependencies and load API keys.

## 1.1 installing dependencies

In [1]:
import warnings
warnings.filterwarnings('ignore')

# !pip install -r requirements.txt

## 1.2 Loading the OpenAI API key

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Section 2 : Data Chunking
loading the mock data and split it into chunks.

In [3]:
from rag_utils import DataProcessor

# Initialize the Data processor
processor = DataProcessor()

# Load and split the document
chunks = processor.load_and_split(directory_path="data/", chunk_size=800, chunk_overlap=120)

print(f"Total chunks created: {len(chunks)}")

sources = set([chunk.metadata.get("source") for chunk in chunks])
print(f"Sources loaded: {sources}")

100%|██████████| 1/1 [00:00<00:00, 341.86it/s]

Total chunks created: 14
Sources loaded: {'data/certifications.txt', 'data/linkedin_profile.txt', 'data/github_projects.md', 'data/Mohammad_CV.pdf'}


# Section 3 : Vector Database & Retriever
Convert the text chunks into embeddings using HuggingFace's `all-MiniLM-L6-v2` model and store them in a ChromaDB vector store.

In [4]:
# Create vector store
vectorstore = processor.create_vector_store(chunks, persist_directory="./chroma_db")
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

# Section 4 : Building the RAG Pipeline
inject our retriever into the `RAGPipeline` class. This class handles the prompt engineering and the LangChain Expression Language (LCEL) to ensure the LLM answers based *only* on the retrieved context and cites its sources.

In [5]:
from rag_chain import RAGPipeline

# Initialize the RAG pipeline with the retriever
rag_pipeline = RAGPipeline(retriever=retriever)
rag_chain = rag_pipeline.build()

# Section 5 : Inference

In [6]:
def query_rag(question: str):
    print(f"Question: {question}\n")

    # Invoke the chain
    result = rag_chain.invoke(question)

    print(f"Answer:\n{result['answer']}\n")

    print("Source Documents Used:")
    for i, doc in enumerate(result["context"]):
        # Print the source file name along with the content
        source_name = doc.metadata.get("source", "Unknown")
        print(f"  [Source {i+1} - {source_name}]:")
        print(f"  {doc.page_content.strip()}\n")
    print("-" * 50)

## Test 1: Specific skill question (Tests Skills & Languages section)

In [7]:
query_rag("What are Mohammad's main programming languages?")

Question: What are Mohammad's main programming languages?

Answer:
Mohammad's main programming languages are Python, Swift, C++, Java, and Dart.

Source Documents Used:
  [Source 1 - data/linkedin_profile.txt]:
  Skills

Programming languages: Python, Swift, C++, Java, Dart.
Mobile engineering: iOS, Flutter, BLoC, state management, mobile-AI integration.
AI and data: machine learning, deep learning, computer vision, feature engineering, LLMs, RAG, agents, model optimization, ONNX.
Backend: REST APIs, FastAPI, authentication, system architecture.
Data and cloud: Supabase, Firebase, SQLite, Oracle.
Engineering tools: Git, GitHub, Docker, CI/CD.

Languages
Arabic is Mohammad's native language. English is used professionally and academically.

  [Source 2 - data/Mohammad_CV.pdf]:
  Arab American University
GPA : 3.21 / 4.0
06/2019 – 04/2025
Jenin, Palestine
TECHNICAL SKILLS
•Programming Languages: Python, Swift, C++, Java.
•Mobile Development: iOS Development (Swift), Cross-Platform (Flutt

## Test 2: Experience question (Tests Professional Experience section)

In [8]:
query_rag("Where did Mohammad work and what did he do there?")

Question: Where did Mohammad work and what did he do there?

Answer:
Mohammad has held two internships:

- **AI Engineer Intern at Cellula Technologies** (September 2026 – Present, Remote) – He worked on practical AI problems using real‑world datasets, contributing to LLM and retrieval‑augmented generation systems, voice‑enabled AI agents, and computer vision models. He built end‑to‑end AI pipelines with Docker and CI/CD, collaborating with engineering teams on deployment, debugging, and scaling models. (Also noted in his CV as developing production AI solutions including LLMs, RAG systems, voice‑enabled AI agents, and computer vision models.)

- **Mobile Development Intern at ASAL Technologies** (October 2024 – April 2025, Rawabi, Palestine) – He re‑engineered the application architecture to improve stability, scalability, and performance across core workflows, and developed interactive community features and real‑time messaging, ensuring reliability through comprehensive testing and 

## Test 3: GitHub question (Tests Projects section)

In [9]:
query_rag("What kind of projects has Mohammad worked on?")

Question: What kind of projects has Mohammad worked on?

Answer:
Based on the provided context, Mohammad has worked on the following specific projects and project‑related work:

- **Plantie App** – a full‑stack mobile & AI architecture Flutter app built using BLoC for state management (listed under the PROJECTS section of his CV).  
- During his **AI Engineer Internship at Cellula Technologies**, he contributed to AI‑focused projects such as large language models (LLMs), retrieval‑augmented generation (RAG) systems, voice‑enabled AI agents, and computer‑vision models for real‑world datasets.  
- During his **Mobile Development Internship at ASAL Technologies**, he worked on projects that re‑engineered the application architecture to improve stability, scalability, and performance, and developed interactive community features and real‑time messaging functionality.  

These are the named project and the concrete project‑type activities mentioned in the context.

Source Documents Used:
  

## Test 4: Certifications question (Tests Certifications section)

In [10]:
query_rag("What certifications does Mohammad have?")

Question: What certifications does Mohammad have?

Answer:
Mohammad has the following certifications:

- **PIP Professional Development Curriculum** – Palestinian Internship Program, completed in 2024 (Credential ID: PIP-PDC-2024-0822)  
- **The Complete 2022 Flutter & Dart Development Course** – Udemy, instructed by Abdullah Mansour, completed on 18 June 2022 (Credential ID: UD-FLT-2022-0618)  
- **Native iOS Development Training** – ASAL Technologies, completed in 2025 (Credential ID: ASAL-IOS-2025-0703)

Source Documents Used:
  [Source 1 - data/linkedin_profile.txt]:
  Skills

Programming languages: Python, Swift, C++, Java, Dart.
Mobile engineering: iOS, Flutter, BLoC, state management, mobile-AI integration.
AI and data: machine learning, deep learning, computer vision, feature engineering, LLMs, RAG, agents, model optimization, ONNX.
Backend: REST APIs, FastAPI, authentication, system architecture.
Data and cloud: Supabase, Firebase, SQLite, Oracle.
Engineering tools: Git, GitHu

## Test 5: Out-of-scope question (testing hallucination guardrails)

In [11]:
query_rag("What is Mohammad's favorite food?")

Question: What is Mohammad's favorite food?

Answer:
I don't have enough information to answer that.

Source Documents Used:
  [Source 1 - data/linkedin_profile.txt]:
  Skills

Programming languages: Python, Swift, C++, Java, Dart.
Mobile engineering: iOS, Flutter, BLoC, state management, mobile-AI integration.
AI and data: machine learning, deep learning, computer vision, feature engineering, LLMs, RAG, agents, model optimization, ONNX.
Backend: REST APIs, FastAPI, authentication, system architecture.
Data and cloud: Supabase, Firebase, SQLite, Oracle.
Engineering tools: Git, GitHub, Docker, CI/CD.

Languages
Arabic is Mohammad's native language. English is used professionally and academically.

  [Source 2 - data/github_projects.md]:
  A Swift iOS chat application using Firebase for real-time messaging.

---

## Mobile Price Classification

A Jupyter Notebook project exploring classification of mobile device price categories using machine learning.

---

## littlelemon

A small Swift

In [12]:
query_rag("What is Mohammad's summary in his cv ?")

Question: What is Mohammad's summary in his cv ?

Answer:
Computer Engineer with a strong foundation in software engineering and problem solving skills. Specialized in mobile development and AI, with expertise in driving the complete machine learning pipeline. Adept at seamlessly integrating models into end-to-end applications, whether through optimized local on‑device inference or by scalable APIs. Thrives in dynamic technical environments and quickly adapts to new tech stacks, with a continuous focus on delivering reliable, high‑impact software solutions.

Source Documents Used:
  [Source 1 - data/linkedin_profile.txt]:
  Mohammad Bassam Nassar
Computer Engineer | ML & Mobile Developer | AI Engineer
Jenin, Palestine
m.nassar4004@gmail.com
+970 595243439


Experience

AI Engineer Intern at Cellula Technologies
September 2026 to Present, Remote

Working on practical AI problems using real-world datasets. Contributing to LLM and retrieval-augmented generation systems, voice-enabled AI a

In [13]:
query_rag("What is Mohammad's email ?")

Question: What is Mohammad's email ?

Answer:
m.nassar4004@gmail.com

Source Documents Used:
  [Source 1 - data/linkedin_profile.txt]:
  Skills

Programming languages: Python, Swift, C++, Java, Dart.
Mobile engineering: iOS, Flutter, BLoC, state management, mobile-AI integration.
AI and data: machine learning, deep learning, computer vision, feature engineering, LLMs, RAG, agents, model optimization, ONNX.
Backend: REST APIs, FastAPI, authentication, system architecture.
Data and cloud: Supabase, Firebase, SQLite, Oracle.
Engineering tools: Git, GitHub, Docker, CI/CD.

Languages
Arabic is Mohammad's native language. English is used professionally and academically.

  [Source 2 - data/linkedin_profile.txt]:
  Mohammad Bassam Nassar
Computer Engineer | ML & Mobile Developer | AI Engineer
Jenin, Palestine
m.nassar4004@gmail.com
+970 595243439


Experience

AI Engineer Intern at Cellula Technologies
September 2026 to Present, Remote

Working on practical AI problems using real-world datase